# PyGAGE, explained from the beginning

PyGAGE tests **groups of genes** instead of looking at only one gene at a time.

A gene set can represent a pathway. If many genes in that pathway move together, PyGAGE can report that pathway as enriched.

This notebook covers:

1. a tiny example you can understand by looking at it;
2. all three tests and both ways of combining results;
3. the new large repository data table;
4. saved charts, command-line outputs, and numerical R GAGE checks.

In [ ]:
from pathlib import Path
import json
import os
import sys

HERE = Path.cwd().resolve()
ROOT = next(
    (
        folder for folder in (HERE, *HERE.parents)
        if (folder / ".venv").exists() and (folder / "scripts").exists()
    ),
    HERE,
)

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))
print("Validation folder:", ROOT)

In [ ]:
import numpy as np
import polars as pl
import pygage
from pygage import GAGEAnalysis, GAGEPreparation, load_gmt, read_matrix
from pygage.visualization_utils import EnrichmentPlots
from IPython.display import Image, display

print("PyGAGE version:", pygage.__version__)
print("Python version:", sys.version.split()[0])

## Part 1 — Read a very small expression table

- Each **row** is one gene.
- `control_1` and `control_2` are reference samples.
- `treatment_1` and `treatment_2` are treatment samples.
- Genes `g01` through `g10` were designed to increase.
- Genes `g11` through `g20` were designed to decrease.

In [ ]:
expression = read_matrix(ROOT / "data" / "toy_expression.csv")
gene_sets = load_gmt(ROOT / "data" / "toy_sets.gmt")
print("Table shape (genes, columns):", expression.shape)
expression.head(8)

## Part 2 — Turn expression into treatment-minus-control values

PyGAGE uses positions starting at zero. Among the sample columns:

- controls are positions `0` and `1`;
- treatments are positions `2` and `3`.

`comparison="paired"` subtracts each matched control from its matched treatment.

In [ ]:
prepared = GAGEPreparation.prepare_expression(
    expression,
    ref_indices=[0, 1],
    samp_indices=[2, 3],
    comparison="paired",
)
prepared.head(8)

Positive numbers mean the treatment is higher. Negative numbers mean the treatment is lower.

In [ ]:
analysis = GAGEAnalysis()
result = analysis.run_gage(
    prepared,
    gene_sets,
    set_size_range=(5, 50),
    test_method="t-test",
    meta_method="stouffer",
    compute_effect=True,
    leading_edge=True,
)

print("Top pathways that increased:")
display(result["greater"].head())
print("Top pathways that decreased:")
display(result["less"].head())

`p_val` measures evidence before multiple-testing correction. `q_val` is the Benjamini–Hochberg corrected value. Smaller values mean stronger evidence.

## Part 3 — Check every statistic/meta-method combination

PyGAGE currently offers three gene-set tests:

- **t-test:** compares the pathway mean with the background and includes variance;
- **z-test:** a PAGE-style standardized mean comparison;
- **KS test:** compares ranked distributions and is less tied to a normal-distribution assumption.

It can combine sample-level evidence with:

- **Stouffer:** combines normalized p-value scores;
- **Fisher:** combines the logarithms of p-values.

In [ ]:
method_rows = []
for test_method in ("t-test", "z-test", "ks-test"):
    for meta_method in ("stouffer", "fisher"):
        tested = GAGEAnalysis().run_gage(
            prepared,
            gene_sets,
            set_size_range=(5, 50),
            test_method=test_method,
            meta_method=meta_method,
        )
        method_rows.append({
            "test": test_method,
            "combine": meta_method,
            "top_greater": tested["greater"]["gene_set"][0],
            "top_less": tested["less"]["gene_set"][0],
            "tested_sets": tested["greater"].height,
        })
pl.DataFrame(method_rows)

## Part 4 — What the PyGAGE charts mean

- **Bubble plot:** one bubble per pathway. Horizontal position is the pathway statistic, color is significance, and bubble size is the number of genes.
- **Enrichment heatmap:** compares pathway statistics across conditions.
- **Running enrichment plot:** walks down a ranked gene list and shows where pathway genes cluster.
- **Pathway gene-color chart:** gives every pathway gene a fold-change color that can be handed to a pathway renderer.

In [ ]:
bubble = ROOT / "results" / "pygage" / "toy_bubble.png"
running = ROOT / "results" / "pygage" / "toy_running_enrichment.png"
display(Image(filename=str(bubble), width=700))
display(Image(filename=str(running), width=700))

## Part 5 — The new large repository data table

The newest PyGAGE repository commit adds one file. It has 19,469 genes and 198 TCGA-style sample columns.

The sample endings identify 184 `.01`, 13 `.11`, and one `.06` sample. The careful classical design uses the **13 patient-matched `.11`/`.01` pairs**. It does not incorrectly pair all 184 primary samples with 13 normal samples.

The repository filename says `GDS3627`, but its columns use TCGA identifiers. Confirm the dataset's exact provenance before describing it in a publication.

In [ ]:
new_expression = read_matrix(
    ROOT / "data" / "upstream_pygage" / "GDS3627_exp_formatted.csv"
)
sample_columns = new_expression.columns[1:]

def patient(sample):
    return ".".join(sample.split(".")[:3])

primary = {
    patient(sample): sample
    for sample in sample_columns
    if sample.endswith(".01")
}
normal = [
    sample for sample in sample_columns
    if sample.endswith(".11") and patient(sample) in primary
]
tumor = [primary[patient(sample)] for sample in normal]

design = pl.DataFrame({
    "patient": [patient(sample) for sample in normal],
    "normal_reference": normal,
    "matched_primary_sample": tumor,
})
print("Expression shape:", new_expression.shape)
print("Matched pairs:", design.height)
design

In [ ]:
ref_indices = [sample_columns.index(sample) for sample in normal]
samp_indices = [sample_columns.index(sample) for sample in tumor]

new_prepared = GAGEPreparation.prepare_expression(
    new_expression,
    ref_indices=ref_indices,
    samp_indices=samp_indices,
    comparison="paired",
    input_logged=True,
)
upstream_sets = json.loads(
    (ROOT / "data" / "upstream_pygage" / "kegg_gs.json").read_text()
)
new_result = GAGEAnalysis().run_gage(
    new_prepared,
    upstream_sets,
    test_method="t-test",
    meta_method="stouffer",
)
print("Prepared shape:", new_prepared.shape)
print("Top greater pathways:")
display(new_result["greater"].head(8))
print("Top less pathways:")
display(new_result["less"].head(8))

In [ ]:
display(Image(
    filename=str(ROOT / "results" / "pygage" / "new_dataset_greater_bubble.png"),
    width=850,
))

## Part 6 — Numerical check against R GAGE

The repository includes reference tables created by R GAGE. The validation joined pathways by name and checked t/Stouffer, z/Stouffer, and t/Fisher results.

In [ ]:
parity = pl.read_csv(ROOT / "results" / "pygage" / "r_gage_parity.csv")
parity

## Part 7 — Command line checks

These equivalent terminal workflows were also executed:

```bash
pygage run data/toy_expression.csv -g data/toy_sets.gmt           -o results/pygage/cli_run.csv --ref 0,1 --samp 2,3 --min-size 5

pygage go data/toy_annotations.gaf -o results/pygage/cli_go.json           --obo data/toy_go.obo --aspect BP --propagate

pygage compare results/pygage/toy_greater.csv results/pygage/toy_less.csv           -o results/pygage/cli_compare.csv --names greater,less
```

In [ ]:
pl.read_csv(ROOT / "results" / "pygage" / "cli_run.csv").head()

## PyGAGE conclusion

The main classical PyGAGE engine, supported inputs, six statistic/meta-method combinations, result helpers, general charts, three CLI workflows, new 13-pair data analysis, and packaged R GAGE numerical regressions all executed.

See the full report for the exact test count and current compatibility notes.